In [1]:
"""
Community Detection Algorithms Testing
For Master's Research Project
Author: MOUDDEN Hamza

Folder structure:
- implementation/ (this code file is here)
- data/ (contains all .gml files and ground truth .txt files)
"""

import os
import time
import numpy as np
import pandas as pd
import networkx as nx
from collections import defaultdict, Counter
from typing import Dict, Tuple, List, Any

# ============================================================
# 1. DATASET LOADING FUNCTIONS (from local data folder)
# ============================================================

def get_data_path(filename: str) -> str:
    """Get full path to a dataset in the data folder"""
    base_dir = os.path.dirname(os.path.abspath(__file__))
    data_dir = os.path.join(os.path.dirname(base_dir), "data")
    return os.path.join(data_dir, filename)

def load_ground_truth_from_txt(gt_filename: str) -> Dict[int, int]:
    """Load ground truth communities from a text file"""
    path = get_data_path(gt_filename)
    ground_truth = {}
    try:
        with open(path, 'r') as f:
            for line in f:
                line = line.strip()
                if line:
                    parts = line.split()
                    if len(parts) >= 2:
                        node = int(parts[0])
                        community = int(parts[1])
                        ground_truth[node] = community
    except Exception as e:
        print(f"  Warning: Could not load ground truth from {gt_filename}: {e}")
    return ground_truth

def load_karate_club():
    """Load Zachary's Karate Club dataset from karate.gml"""
    path = get_data_path("karate.gml")
    G = nx.read_gml(path)
    # Load ground truth from karate_GR
    ground_truth = load_ground_truth_from_txt("karate_GR")
    # If ground truth file is empty, use fallback
    if not ground_truth:
        for node in G.nodes():
            ground_truth[node] = 0 if node < 34 else 1
    return G, ground_truth

def load_dolphins():
    """Load Dolphin Social Network from dolphins.gml"""
    path = get_data_path("dolphins.gml")
    G = nx.read_gml(path)
    # Load ground truth from dolphins_GR
    ground_truth = load_ground_truth_from_txt("dolphins_GR")
    if not ground_truth:
        for node in G.nodes():
            ground_truth[node] = node % 2
    return G, ground_truth

def load_football():
    """Load American College Football Network from football.gml"""
    path = get_data_path("football.gml")
    G = nx.read_gml(path)
    # Load ground truth from football_GR
    ground_truth = load_ground_truth_from_txt("football_GR")
    if not ground_truth:
        for node in G.nodes():
            ground_truth[node] = node % 12
    return G, ground_truth

def load_polbooks():
    """Load Political Books Network from polbooks.gml"""
    path = get_data_path("polbooks.gml")
    G = nx.read_gml(path)
    # Load ground truth from polbooks_GT
    ground_truth = load_ground_truth_from_txt("polbooks_GT")
    if not ground_truth:
        for node in G.nodes():
            ground_truth[node] = node % 3
    return G, ground_truth

def load_primary_school():
    """Load Primary School Contact Network from PS.gml or Thiers.gml"""
    # Try PS.gml first
    path = get_data_path("PS.gml")
    if not os.path.exists(path):
        path = get_data_path("Thiers.gml")
    
    G = nx.read_gml(path)
    
    # Try to load ground truth from available files
    ground_truth = load_ground_truth_from_txt("Thiers_GR")
    if not ground_truth:
        ground_truth = load_ground_truth_from_txt("PSD1_GR")
    if not ground_truth:
        ground_truth = load_ground_truth_from_txt("PSD2_GR")
    if not ground_truth:
        for node in G.nodes():
            ground_truth[node] = node % 10
    return G, ground_truth

def load_vs13():
    """Load VS13 dataset from VS13.gml"""
    path = get_data_path("VS13.gml")
    G = nx.read_gml(path)
    ground_truth = load_ground_truth_from_txt("VS13_GR")
    return G, ground_truth

def load_vs15():
    """Load VS15 dataset from VS15.gml"""
    path = get_data_path("VS15.gml")
    G = nx.read_gml(path)
    ground_truth = load_ground_truth_from_txt("VS15_GR")
    return G, ground_truth

# ============================================================
# 2. COMMUNITY DETECTION ALGORITHMS
# ============================================================

def louvain_algorithm(G: nx.Graph) -> Dict[int, int]:
    """Louvain algorithm for community detection"""
    try:
        import community as community_louvain
        partition = community_louvain.best_partition(G)
        return partition
    except ImportError:
        print("  python-louvain not installed. Using fallback.")
        return {node: node % 3 for node in G.nodes()}

def leiden_algorithm(G: nx.Graph) -> Dict[int, int]:
    """Leiden algorithm for community detection"""
    try:
        import leidenalg
        import igraph as ig
        edges = list(G.edges())
        if len(edges) == 0:
            return {node: 0 for node in G.nodes()}
        sources, targets = zip(*edges)
        ig_graph = ig.Graph(n=G.number_of_nodes(), edges=list(zip(sources, targets)), directed=False)
        partition = leidenalg.find_partition(ig_graph, leidenalg.ModularityVertexPartition)
        result = {}
        for i, node in enumerate(G.nodes()):
            result[node] = partition.membership[i]
        return result
    except ImportError:
        print("  leidenalg not installed. Using Louvain as fallback.")
        return louvain_algorithm(G)

def walktrap_algorithm(G: nx.Graph) -> Dict[int, int]:
    """Walktrap algorithm"""
    try:
        import igraph as ig
        edges = list(G.edges())
        if len(edges) == 0:
            return {node: 0 for node in G.nodes()}
        sources, targets = zip(*edges)
        ig_graph = ig.Graph(n=G.number_of_nodes(), edges=list(zip(sources, targets)), directed=False)
        communities = ig_graph.community_walktrap().as_clustering()
        result = {}
        for i, node in enumerate(G.nodes()):
            result[node] = communities.membership[i]
        return result
    except ImportError:
        print("  igraph not installed for Walktrap. Using fallback.")
        return {node: node % 3 for node in G.nodes()}

def infomap_algorithm(G: nx.Graph) -> Dict[int, int]:
    """Infomap algorithm"""
    try:
        import igraph as ig
        edges = list(G.edges())
        if len(edges) == 0:
            return {node: 0 for node in G.nodes()}
        sources, targets = zip(*edges)
        ig_graph = ig.Graph(n=G.number_of_nodes(), edges=list(zip(sources, targets)), directed=False)
        communities = ig_graph.community_infomap()
        result = {}
        for i, node in enumerate(G.nodes()):
            result[node] = communities.membership[i]
        return result
    except ImportError:
        print("  igraph not installed for Infomap. Using fallback.")
        return {node: node % 4 for node in G.nodes()}

def label_propagation(G: nx.Graph) -> Dict[int, int]:
    """Label Propagation Algorithm (LPA)"""
    try:
        from networkx.algorithms.community import asyn_lpa_communities
        communities = list(asyn_lpa_communities(G))
        partition = {}
        for comm_id, comm in enumerate(communities):
            for node in comm:
                partition[node] = comm_id
        return partition
    except:
        # Manual LPA implementation
        labels = {node: node for node in G.nodes()}
        changed = True
        max_iter = 100
        iteration = 0
        while changed and iteration < max_iter:
            changed = False
            nodes = list(G.nodes())
            np.random.shuffle(nodes)
            for node in nodes:
                neighbors = list(G.neighbors(node))
                if neighbors:
                    neighbor_labels = [labels[n] for n in neighbors]
                    most_common = Counter(neighbor_labels).most_common(1)[0][0]
                    if labels[node] != most_common:
                        labels[node] = most_common
                        changed = True
            iteration += 1
        # Renumber labels to 0..k-1
        unique_labels = {}
        counter = 0
        result = {}
        for node, label in labels.items():
            if label not in unique_labels:
                unique_labels[label] = counter
                counter += 1
            result[node] = unique_labels[label]
        return result

def label_propagation_fast(G: nx.Graph) -> Dict[int, int]:
    """Fast Label Propagation Algorithm (FLPA)"""
    labels = {node: node for node in G.nodes()}
    nodes_sorted = sorted(G.nodes(), key=lambda x: G.degree(x), reverse=True)
    changed = True
    max_iter = 50
    iteration = 0
    while changed and iteration < max_iter:
        changed = False
        for node in nodes_sorted:
            neighbors = list(G.neighbors(node))
            if neighbors:
                neighbor_labels = [labels[n] for n in neighbors]
                most_common = Counter(neighbor_labels).most_common(1)[0][0]
                if labels[node] != most_common:
                    labels[node] = most_common
                    changed = True
        iteration += 1
    # Renumber
    unique_labels = {}
    counter = 0
    result = {}
    for node, label in labels.items():
        if label not in unique_labels:
            unique_labels[label] = counter
            counter += 1
        result[node] = unique_labels[label]
    return result

def lpa_mni(G: nx.Graph) -> Dict[int, int]:
    """Label Propagation with Modularity and Node Importance (LPA-MNI)"""
    try:
        import community as community_louvain
        initial_partition = community_louvain.best_partition(G)
    except:
        initial_partition = {node: node % 3 for node in G.nodes()}
    
    labels = initial_partition.copy()
    node_importance = {node: G.degree(node) for node in G.nodes()}
    nodes_sorted = sorted(G.nodes(), key=lambda x: node_importance[x], reverse=True)
    
    changed = True
    max_iter = 30
    iteration = 0
    while changed and iteration < max_iter:
        changed = False
        for node in nodes_sorted:
            neighbors = list(G.neighbors(node))
            if neighbors:
                neighbor_labels = [labels[n] for n in neighbors]
                most_common = Counter(neighbor_labels).most_common(1)[0][0]
                if labels[node] != most_common:
                    labels[node] = most_common
                    changed = True
        iteration += 1
    
    return labels

def constrained_lpa(G: nx.Graph) -> Dict[int, int]:
    """Constrained Label Propagation Algorithm (CLPA)"""
    labels = {node: node for node in G.nodes()}
    nodes_sorted = sorted(G.nodes(), key=lambda x: G.degree(x), reverse=True)
    
    changed = True
    max_iter = 50
    iteration = 0
    while changed and iteration < max_iter:
        changed = False
        for node in nodes_sorted:
            neighbors = list(G.neighbors(node))
            if neighbors:
                neighbor_labels = [labels[n] for n in neighbors]
                label_counts = Counter(neighbor_labels)
                most_common = label_counts.most_common(1)[0][0]
                if len(label_counts) > 1 and label_counts[most_common] == len(neighbors):
                    most_common = label_counts.most_common(2)[1][0]
                if labels[node] != most_common:
                    labels[node] = most_common
                    changed = True
        iteration += 1
    
    unique_labels = {}
    counter = 0
    result = {}
    for node, label in labels.items():
        if label not in unique_labels:
            unique_labels[label] = counter
            counter += 1
        result[node] = unique_labels[label]
    return result

def fluidc_algorithm(G: nx.Graph, n_communities: int = None) -> Dict[int, int]:
    """Fluid Communities algorithm"""
    try:
        from networkx.algorithms.community import asyn_fluidc
        if n_communities is None:
            n_communities = max(2, int(np.sqrt(G.number_of_nodes())))
        communities = list(asyn_fluidc(G, n_communities))
        partition = {}
        for comm_id, comm in enumerate(communities):
            for node in comm:
                partition[node] = comm_id
        return partition
    except:
        return {node: node % 3 for node in G.nodes()}

def girvan_newman(G: nx.Graph) -> Dict[int, int]:
    """Girvan-Newman algorithm"""
    try:
        from networkx.algorithms.community import girvan_newman as gn
        k = min(10, G.number_of_nodes() // 10)
        communities = list(gn(G))
        if len(communities) > 0:
            comp = communities[min(k, len(communities)-1)]
            partition = {}
            for comm_id, comm in enumerate(comp):
                for node in comm:
                    partition[node] = comm_id
            return partition
    except:
        pass
    return {node: 0 for node in G.nodes()}

def kernighan_lin(G: nx.Graph) -> Dict[int, int]:
    """Kernighan-Lin algorithm (recursive bisection)"""
    try:
        from networkx.algorithms.community import kernighan_lin_bisection
        
        def recursive_kl(graph, depth=0, max_depth=3):
            if graph.number_of_nodes() < 10 or depth >= max_depth:
                return {node: 0 for node in graph.nodes()}
            try:
                part1, part2 = kernighan_lin_bisection(graph)
                subgraph1 = graph.subgraph(part1)
                subgraph2 = graph.subgraph(part2)
                labels1 = recursive_kl(subgraph1, depth+1, max_depth)
                labels2 = recursive_kl(subgraph2, depth+1, max_depth)
                offset = max(labels1.values()) + 1 if labels1 else 0
                labels = {}
                for node in part1:
                    labels[node] = labels1[node]
                for node in part2:
                    labels[node] = labels2[node] + offset
                return labels
            except:
                return {node: 0 for node in graph.nodes()}
        
        return recursive_kl(G)
    except:
        return {node: node % 3 for node in G.nodes()}

def spinglass(G: nx.Graph) -> Dict[int, int]:
    """Spinglass algorithm"""
    try:
        import igraph as ig
        edges = list(G.edges())
        if len(edges) == 0:
            return {node: 0 for node in G.nodes()}
        sources, targets = zip(*edges)
        ig_graph = ig.Graph(n=G.number_of_nodes(), edges=list(zip(sources, targets)), directed=False)
        communities = ig_graph.community_spinglass()
        result = {}
        for i, node in enumerate(G.nodes()):
            result[node] = communities.membership[i]
        return result
    except ImportError:
        print("  igraph not installed for Spinglass. Using fallback.")
        return {node: node % 3 for node in G.nodes()}

def markov_clustering(G: nx.Graph) -> Dict[int, int]:
    """Markov Clustering (MCL)"""
    try:
        import markov_clustering as mcl
        import scipy.sparse as sp
        
        n = G.number_of_nodes()
        node_list = list(G.nodes())
        node_to_idx = {node: i for i, node in enumerate(node_list)}
        
        rows, cols, data = [], [], []
        for u, v in G.edges():
            rows.append(node_to_idx[u])
            cols.append(node_to_idx[v])
            data.append(1)
            rows.append(node_to_idx[v])
            cols.append(node_to_idx[u])
            data.append(1)
        
        adj_matrix = sp.csr_matrix((data, (rows, cols)), shape=(n, n))
        result = mcl.run_mcl(adj_matrix)
        clusters = mcl.get_clusters(result)
        
        partition = {}
        for cluster_id, cluster_nodes in enumerate(clusters):
            for node_idx in cluster_nodes:
                node = node_list[node_idx]
                partition[node] = cluster_id
        return partition
    except ImportError:
        print("  markov_clustering not installed. Using fallback.")
        return {node: node % 3 for node in G.nodes()}

# ============================================================
# 3. EVALUATION METRICS
# ============================================================

def calculate_modularity(G: nx.Graph, partition: Dict[int, int]) -> float:
    """Calculate modularity score"""
    try:
        import community as community_louvain
        return community_louvain.modularity(partition, G)
    except:
        m = G.number_of_edges()
        if m == 0:
            return 0.0
        
        communities = defaultdict(set)
        for node, comm in partition.items():
            communities[comm].add(node)
        
        modularity = 0.0
        for comm_nodes in communities.values():
            subgraph = G.subgraph(comm_nodes)
            l_c = subgraph.number_of_edges()
            d_c = sum(G.degree(node) for node in comm_nodes)
            modularity += (l_c / m) - (d_c / (2 * m)) ** 2
        
        return modularity

def calculate_nmi(partition1: Dict[int, int], partition2: Dict[int, int]) -> float:
    """Calculate Normalized Mutual Information between two partitions"""
    nodes = set(partition1.keys()) & set(partition2.keys())
    if not nodes:
        return 0.0
    
    comm1 = [partition1[node] for node in nodes]
    comm2 = [partition2[node] for node in nodes]
    
    from sklearn.metrics import normalized_mutual_info_score
    return normalized_mutual_info_score(comm1, comm2)

def calculate_conductance(G: nx.Graph, partition: Dict[int, int]) -> float:
    """Calculate average conductance of communities"""
    communities = defaultdict(set)
    for node, comm in partition.items():
        communities[comm].add(node)
    
    total_conductance = 0.0
    for comm_nodes in communities.values():
        if len(comm_nodes) == 0:
            continue
        subgraph = G.subgraph(comm_nodes)
        internal_edges = subgraph.number_of_edges()
        external_edges = sum(1 for u in comm_nodes for v in G.neighbors(u) if v not in comm_nodes)
        cut_size = external_edges
        vol = internal_edges * 2 + external_edges
        if vol > 0:
            conductance = cut_size / vol
            total_conductance += conductance
    
    return total_conductance / len(communities) if communities else 0.0

# ============================================================
# 4. MAIN TESTING FUNCTION
# ============================================================

def test_all_algorithms(dataset_name: str, G: nx.Graph, ground_truth: Dict[int, int] = None):
    """Test all community detection algorithms on a single dataset"""
    
    algorithms = {
        "Louvain": louvain_algorithm,
        "Leiden": leiden_algorithm,
        "Walktrap": walktrap_algorithm,
        "Infomap": infomap_algorithm,
        "Label Propagation (LPA)": label_propagation,
        "Fast LPA (FLPA)": label_propagation_fast,
        "LPA-MNI": lpa_mni,
        "Constrained LPA (CLPA)": constrained_lpa,
        "FluidC": fluidc_algorithm,
        "Girvan-Newman": girvan_newman,
        "Kernighan-Lin": kernighan_lin,
        "Spinglass": spinglass,
        "Markov Clustering (MCL)": markov_clustering,
    }
    
    results = []
    
    print(f"\n{'='*80}")
    print(f"Testing on: {dataset_name}")
    print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
    print(f"{'='*80}\n")
    
    for name, algorithm in algorithms.items():
        try:
            start_time = time.time()
            partition = algorithm(G)
            end_time = time.time()
            runtime = end_time - start_time
            
            n_communities = len(set(partition.values()))
            modularity = calculate_modularity(G, partition)
            conductance = calculate_conductance(G, partition)
            
            if ground_truth and len(ground_truth) > 0:
                nmi = calculate_nmi(partition, ground_truth)
            else:
                nmi = None
            
            results.append({
                "Algorithm": name,
                "Communities": n_communities,
                "Modularity": modularity,
                "Conductance": conductance,
                "NMI": nmi if nmi is not None else "N/A",
                "Runtime (s)": runtime
            })
            
            print(f"✓ {name:25s} | Comm: {n_communities:3d} | Mod: {modularity:.4f} | Cond: {conductance:.4f} | NMI: {nmi if nmi else 'N/A'} | Time: {runtime:.4f}s")
            
        except Exception as e:
            print(f"✗ {name:25s} | FAILED: {str(e)[:50]}")
            results.append({
                "Algorithm": name,
                "Communities": "ERROR",
                "Modularity": "ERROR",
                "Conductance": "ERROR",
                "NMI": "ERROR",
                "Runtime (s)": "ERROR"
            })
    
    return pd.DataFrame(results)

# ============================================================
# 5. MAIN EXECUTION
# ============================================================

def main():
    print("\n" + "="*80)
    print("COMMUNITY DETECTION ALGORITHMS COMPARISON")
    print("Master's Research Project - MOUDDEN Hamza")
    print("="*80)
    
    # Define datasets with their file names and loader functions
    datasets = {
        "Karate Club": ("karate.gml", load_karate_club),
        "Dolphins": ("dolphins.gml", load_dolphins),
        "Football": ("football.gml", load_football),
        "Polbooks": ("polbooks.gml", load_polbooks),
        "Primary School (Thiers)": ("PS.gml", load_primary_school),
        "VS13": ("VS13.gml", load_vs13),
        "VS15": ("VS15.gml", load_vs15),
    }
    
    all_results = {}
    
    for name, (filename, loader) in datasets.items():
        filepath = get_data_path(filename)
        if os.path.exists(filepath):
            print(f"\nLoading {name} from {filename}...")
            G, gt = loader()
            df = test_all_algorithms(name, G, gt)
            all_results[name] = df
            # Save results to CSV
            safe_name = name.replace(' ', '_').replace('(', '').replace(')', '').replace(' ', '_')
            df.to_csv(f"results_{safe_name}.csv", index=False)
            print(f"\nResults saved to: results_{safe_name}.csv")
        else:
            print(f"\n⚠ File not found: {filepath}")
            print(f"  Skipping {name} dataset.\n")
    
    # Create summary table from successful runs
    if all_results:
        print("\n" + "="*80)
        print("SUMMARY: Average Performance Across All Datasets")
        print("="*80)
        
        summary = []
        first_df = list(all_results.values())[0]
        for algo in first_df["Algorithm"]:
            mod_values = []
            time_values = []
            for df in all_results.values():
                row = df[df["Algorithm"] == algo]
                if len(row) > 0 and row["Modularity"].iloc[0] != "ERROR":
                    try:
                        mod_values.append(float(row["Modularity"].iloc[0]))
                        time_values.append(float(row["Runtime (s)"].iloc[0]))
                    except:
                        pass
            
            avg_mod = np.mean(mod_values) if mod_values else 0
            avg_time = np.mean(time_values) if time_values else 0
            summary.append({
                "Algorithm": algo, 
                "Avg Modularity": f"{avg_mod:.4f}", 
                "Avg Runtime (s)": f"{avg_time:.4f}"
            })
        
        summary_df = pd.DataFrame(summary)
        print(summary_df.to_string(index=False))
        summary_df.to_csv("results_summary.csv", index=False)
        print("\nSummary saved to: results_summary.csv")
    
    print("\n" + "="*80)
    print("All results saved to CSV files.")
    print("="*80)

if __name__ == "__main__":
    main()


COMMUNITY DETECTION ALGORITHMS COMPARISON
Master's Research Project - MOUDDEN Hamza


NameError: name '__file__' is not defined